# Privacy + Org-Switch Regression Test for Databricks

**Problem:** `g.plot()` in Databricks shows a Graphistry login page in the iframe instead of the visualization after upgrading pygraphistry.

**Root cause analysis (local venv comparison):**
- Privacy defaults are **identical** across 0.44.1, 0.45.9, and 0.50.6 — privacy is NOT the cause.
- The **real regression** in 0.50.6 is:
  1. `active_organization` in SSO response is now **required** (was optional/silent in 0.44.1/0.45.9)
  2. New `_switch_org()` call after SSO login hits `/api/v2/o/{org}/switch/`
  3. API version locked to v3 only (was v1 default + v3 in older versions)

**Client confirms:** David Hutchinson (DBT) reports 0.44.1 works, **0.45.9 is the last working version**, 0.50.5 breaks.

**Test matrix:**
| Test | Privacy | Expected |
|------|---------|----------|
| A | None (server default) | Depends on server config |
| B | `mode='public'` | Should render without auth |
| C | `mode='private'` | May show login page in iframe |

Plus: org-switch diagnostics, active_organization check, SSO response inspection.

**Targets:** `graphistry-dev.grph.xyz`, `obsidian-tc.grph.xyz`

In [ ]:
%pip install graphistry==0.50.6 requests
dbutils.library.restartPython()

In [ ]:
import graphistry
import requests
import pandas as pd

# --- Configuration ---
# Toggle between servers:
ACTIVE_SERVER = "graphistry-dev.grph.xyz"
# ACTIVE_SERVER = "obsidian-tc.grph.xyz"
PROTOCOL = "https"

print(f"graphistry version: {graphistry.__version__}")
print(f"Target server:      {PROTOCOL}://{ACTIVE_SERVER}")

In [ ]:
# SSO login (non-blocking)
graphistry.register(
    api=3,
    protocol=PROTOCOL,
    server=ACTIVE_SERVER,
    is_sso_login=True,
    sso_opt_into_type="display",
    sso_timeout=None,
)
print("Click the SSO link above, complete login, then run the next cell.")

In [ ]:
# Retrieve SSO token + inspect SSO response details
token = graphistry.sso_get_token()
assert token is not None and len(token) > 10, f"Token missing or invalid: {repr(token[:20] if token else None)}"
print(f"Token obtained: {token[:20]}...")
print(f"Server: {ACTIVE_SERVER}")

# --- Org-switch diagnostics ---
import inspect
from graphistry.arrow_uploader import ArrowUploader

print("\n=== Org-Switch Regression Diagnostics ===")

# Check if _switch_org exists (0.50.x only)
has_switch = hasattr(ArrowUploader, '_switch_org')
print(f"ArrowUploader._switch_org exists: {has_switch}")

# Check active_organization handling
src = inspect.getsource(ArrowUploader.sso_get_token)
is_required = 'raise Exception' in src and 'active_organization' in src
is_optional = 'active_organization' in src and 'raise Exception' not in src
print(f"active_organization required (raises): {is_required}")
print(f"active_organization optional (silent): {is_optional}")

# Check session state after SSO
session = graphistry.PyGraphistry._config
print(f"\nSession org_name: {getattr(session, 'org_name', 'MISSING')}")
print(f"Session api_version: {getattr(session, 'api_version', 'MISSING')}")
if hasattr(session, '_last_switched_org_token'):
    print(f"Last switched org: {session._last_switched_org_token}")
else:
    print("_last_switched_org_token: NOT PRESENT (pre-0.50.x)")

# Verify token against server
try:
    is_valid = graphistry.verify_token()
    print(f"\nToken valid: {is_valid}")
except Exception as e:
    print(f"\nToken verify error: {e}")

# Test org switch endpoint directly
org = getattr(session, 'org_name', None)
if org and token:
    try:
        switch_url = f"{PROTOCOL}://{ACTIVE_SERVER}/api/v2/o/{org}/switch/"
        resp = requests.post(
            switch_url,
            data={'slug': org},
            headers={'Authorization': f'Bearer {token}'},
            timeout=10,
        )
        print(f"\nOrg switch endpoint ({switch_url}):")
        print(f"  Status: {resp.status_code}")
        print(f"  Response: {resp.text[:200]}")
    except Exception as e:
        print(f"\nOrg switch endpoint error: {e}")
else:
    print(f"\nSkipping org switch test (org={org}, has_token={bool(token)})")

In [ ]:
# Test data
edges = pd.DataFrame({
    "src": ["a", "b", "c", "d", "e"],
    "dst": ["b", "c", "d", "e", "a"],
    "weight": [1, 2, 3, 4, 5],
})

g = graphistry.edges(edges, "src", "dst")
print(f"Graph: {len(edges)} edges")
print(f"g._privacy = {getattr(g, '_privacy', 'MISSING')}")
print(f"session.privacy = {getattr(graphistry.PyGraphistry._config, 'privacy', 'MISSING')}")

In [ ]:
# === Test Matrix: upload 3 datasets with different privacy settings ===
# Use render=False to get URLs without rendering iframes

results = {}

# --- Test A: No privacy set (server default) ---
print("=" * 60)
print("TEST A: No privacy set (server default)")
url_a = g.plot(render=False)
print(f"  URL: {url_a}")
results['A_no_privacy'] = {'url': url_a}

# --- Test B: privacy(mode='public') ---
print("\n" + "=" * 60)
print("TEST B: privacy(mode='public')")
url_b = g.privacy(mode='public').plot(render=False)
print(f"  URL: {url_b}")
results['B_public'] = {'url': url_b}

# --- Test C: privacy(mode='private') ---
print("\n" + "=" * 60)
print("TEST C: privacy(mode='private')")
url_c = g.privacy(mode='private').plot(render=False)
print(f"  URL: {url_c}")
results['C_private'] = {'url': url_c}

# --- Check each URL for login redirect (unauthenticated GET) ---
print("\n" + "=" * 60)
print("Checking URLs without cookies (simulating iframe behavior)...")

for label, data in results.items():
    url = data['url']
    try:
        resp = requests.get(url, allow_redirects=True, timeout=15)
        final_url = resp.url
        is_login = any(x in final_url.lower() for x in ['login', 'signin', 'accounts/login'])
        has_login_form = 'login' in resp.text.lower()[:2000] if resp.text else False
        data['status'] = resp.status_code
        data['final_url'] = final_url
        data['redirected_to_login'] = is_login
        data['login_form_in_body'] = has_login_form
        data['shows_viz'] = not is_login and not has_login_form
        status = 'PASS (viz)' if data['shows_viz'] else 'FAIL (login page)'
        print(f"  {label}: {status}  [HTTP {resp.status_code}, redirect_to_login={is_login}]")
    except Exception as e:
        data['error'] = str(e)
        print(f"  {label}: ERROR - {e}")

In [ ]:
# === Inspect internal state ===
import inspect

print("=== Internal Privacy State ===")
print()

# Session-level privacy
session = graphistry.PyGraphistry._config
print(f"session.privacy: {getattr(session, 'privacy', 'MISSING')}")
print(f"session type:    {type(session).__name__}")
print()

# Graph-level privacy
g_plain = graphistry.edges(edges, "src", "dst")
g_pub = g_plain.privacy(mode='public')
g_priv = g_plain.privacy(mode='private')

print(f"g (no privacy)._privacy:  {g_plain._privacy}")
print(f"g.privacy('public')._privacy:  {g_pub._privacy}")
print(f"g.privacy('private')._privacy: {g_priv._privacy}")
print()

# Check maybe_post_share_link logic
from graphistry.arrow_uploader import ArrowUploader
print("maybe_post_share_link source:")
print(inspect.getsource(ArrowUploader.maybe_post_share_link))
print()

# Check cascade_privacy_settings defaults
if hasattr(ArrowUploader, 'cascade_privacy_settings'):
    src = inspect.getsource(ArrowUploader.cascade_privacy_settings)
    # Extract just the default lines
    for line in src.split('\n'):
        s = line.strip()
        if s.startswith('if mode is None') or s.startswith("mode = "):
            print(f"  cascade default: {s}")

In [ ]:
# === Render actual iframes for visual comparison ===
# In Databricks, displayHTML renders in the notebook output cell.

html_parts = []
html_parts.append("<h2>Privacy / Iframe Visual Comparison</h2>")
html_parts.append('<div style="display: flex; gap: 10px; flex-wrap: wrap;">')

for label, data in results.items():
    url = data.get('url', '')
    status = 'PASS' if data.get('shows_viz') else 'FAIL'
    color = '#2d7d2d' if status == 'PASS' else '#cc3333'
    html_parts.append(f'''
    <div style="border: 2px solid {color}; padding: 5px; min-width: 400px;">
        <h3 style="color: {color};">{label} ({status})</h3>
        <iframe src="{url}" width="400" height="350" style="border:1px solid #ccc;"></iframe>
        <p style="font-size:10px; word-break:break-all;">{url}</p>
    </div>
    ''')

html_parts.append('</div>')
html_out = '\n'.join(html_parts)

# Try Databricks displayHTML, fall back to IPython
try:
    displayHTML(html_out)
except NameError:
    from IPython.display import display, HTML
    display(HTML(html_out))

In [ ]:
# === Summary Table ===

print("=" * 80)
print(f"{'Test':<20} {'Privacy':<15} {'HTTP':<6} {'Login Redirect':<16} {'Login Form':<12} {'Result'}")
print("-" * 80)

for label, data in results.items():
    if 'error' in data:
        print(f"{label:<20} {'?':<15} {'ERR':<6} {'?':<16} {'?':<12} ERROR: {data['error'][:30]}")
    else:
        priv = 'none' if 'no_privacy' in label else ('public' if 'public' in label else 'private')
        result = 'PASS' if data.get('shows_viz') else 'FAIL'
        print(f"{label:<20} {priv:<15} {data.get('status','?'):<6} "
              f"{str(data.get('redirected_to_login','?')):<16} "
              f"{str(data.get('login_form_in_body','?')):<12} {result}")

print("=" * 80)
print()
print("Interpretation:")
print("  - If A (no privacy) FAILS but B (public) PASSES:")
print("    => Server default is 'private'; explicit public fixes it.")
print("  - If A and B both PASS but C (private) FAILS:")
print("    => Only explicit private causes the issue; server default is OK.")
print("  - If all FAIL: iframe sandbox blocks all auth; unrelated to privacy.")
print()
print("=== Regression Root Cause (from local venv analysis) ===")
print()
print("Privacy defaults are IDENTICAL across 0.44.1, 0.45.9, and 0.50.6.")
print("The real breaking changes in 0.50.6 are:")
print("  1. active_organization in SSO response is REQUIRED (was optional)")
print("     - 0.44.1/0.45.9: silently skips if missing")
print("     - 0.50.6: raises Exception('SSO response missing active organization')")
print("  2. NEW _switch_org() call after SSO: POST /api/v2/o/{org}/switch/")
print("     - If this endpoint fails, user may not be in correct org context")
print("  3. API version locked to v3 only (0.44.1 defaulted to v1)")
print()
print("Recommended fix: ensure server's SSO JWT response includes")
print("active_organization.slug, and /api/v2/o/{org}/switch/ works.")
print(f"\nServer tested: {ACTIVE_SERVER}")
print(f"graphistry version: {graphistry.__version__}")